In [ ]:
!dd if=/dev/urandom of=sample_file.bin bs=128 count=1
!openssl sha256 sample_file.bin

In [ ]:
import mmap

import ctypes
import json
import hashlib
import numpy as np
import random
import time

with open('sample_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        # Memory mapped file mm behaves like a bytearray
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash before sort: {sha256_hash}')

        # Treat memory mapped file bytes as integers array in memory.
        # Sorting them.
        int_array = np.frombuffer(mm, dtype=np.uint32)
        print('Bytes as array of garbage 32 unsigned integers')
        print(int_array[:32]) # Unsorted
        int_array.sort()
        print('Bytes as array of sorted 32 unsigned integers')
        print(int_array[:32]) # Sorted
        mm.flush()
        
        # numpy array must be closed before memory mapped file.
        # BufferError: cannot close exported pointers exist
        del int_array

        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash after sort: {sha256_hash}')

In [ ]:
with open('sample_file.bin', 'rb') as f:
    file_content = f.read()
    sha256_hash = hashlib.sha256(file_content).hexdigest()
    print(f'Checking that file content changed indeed: {sha256_hash}')
    
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_READ) as mm:
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash computed from memory mapped file: {sha256_hash}')

        try:
            mm[0] = b'x'
            assert('Cannot write to read-only mmap')
        except TypeError as e:
            #TypeError: mmap can't modify a readonly memory map.
            print('Exception as expected: ', e)

In [ ]:
with open('sample_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        # Memory mapped file mm behaves like a bytearray
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(sha256_hash)

        # Treating buffer as signed integers (~ half of them will be negative)
        IntArrayType = ctypes.c_int * (len(mm) // ctypes.sizeof(ctypes.c_int))
        int_array = IntArrayType.from_buffer(mm)
        
        j = json.dumps({'numbers' : [n for n in int_array]}, indent=4).encode('utf-8')
        print('Previously sorted 32 bit unsigned integers treated as signed')
        print(j)

        # Cannot resize memory mapped file without closing integers arrays still holding a reference.
        # BufferError: cannot close exported pointers exist
        del int_array

        # Deceptively called function "truncate" works just fine to grow the file size forward. 
        f.truncate(len(j))
        
        mm.resize(len(j))
        mm[0:] = j
        mm.flush()

In [ ]:
with open('sample_file.bin', 'r') as f:
    j = json.load(f)
    print('Reading presumably json file back')
    print(j)

In [ ]:
import subprocess

# A tiny C library with one function. It must not need anything from the C runtime, see below.
c_source = r'''
int hello_from_mmap(void) {
    int answer = 0;
    for (int i = 2, end = 14; i < end; i += 2) {
        answer += i;
    }
    return answer; // 42
}
'''
with open('hello_mmap.c', 'w') as f:
    f.write(c_source)

# Compile a position independent shared library
subprocess.run(['cc', '-shared', '-fPIC', '-O2', '-o', 'libhello_mmap.so', 'hello_mmap.c'], check=True)

def run(*args):
    return subprocess.run(args, capture_output=True, check=True, text=True).stdout

def get_function_file_offset(library, function_name):
    # nm prints the function's virtual address, where it lands once the loader maps the library:
    # 0000000000000580 T hello_from_mmap
    address = next(int(line.split()[0], 16) for line in run('nm', '-D', library).splitlines()
                   if line.split()[-1] == function_name)
    # A plain file mapping lays the bytes out as in the file, so turn the address into a file
    # offset through the .text section header: [ 9] .text PROGBITS 0000000000000480 000480 ...
    text = next(line.split(']')[1].split() for line in run('readelf', '-SW', library).splitlines()
                if '] .text ' in line)
    section_address, section_offset = int(text[2], 16), int(text[3], 16)
    offset = address - section_address + section_offset
    print(f'{function_name} address 0x{address:x}, file offset 0x{offset:x}')
    return offset

# Open the .so with mmap, the way the dynamic loader itself maps shared libraries. A plain
# file mapping is not relocated, so its bytes can be read (here the ELF magic).
# dlopen normally mmaps and relocates the library's segments
# but we recklessly call a function from the uninitialized library anyway.
with open('libhello_mmap.so', 'rb') as f:
    # MAP_PRIVATE: writes to the mapping are copy-on-write and never reach the file on disk,
    # so the file can stay open read-only.
    with mmap.mmap(f.fileno(), length=0, flags=mmap.MAP_PRIVATE,
                   prot=mmap.PROT_READ | mmap.PROT_WRITE | mmap.PROT_EXEC) as so:
        print(f'Mapped {len(so)} bytes of libhello_mmap.so, ELF magic {bytes(so[:4])}')

        # Memory protection is opened for read, write and execute
        # because ctypes does not like the idea of taking address of buffer that is not writable.
        # TypeError: underlying buffer is not writable
        mmap_address = ctypes.addressof(ctypes.c_char.from_buffer(so))
        print(f'mmap_address: 0x{mmap_address:x}')

        address = mmap_address + get_function_file_offset('./libhello_mmap.so', 'hello_from_mmap')
        print(f'hello_from_mmap at 0x{address:x} directly from memory mapped file')

        # DANGER! Reckless call from the uninitialized library that would result in a crash
        # if that function needed anything from C runtime or accessed global variables.
        hello_from_mmap = ctypes.CFUNCTYPE(ctypes.c_int)(address)
        result = hello_from_mmap()
        print(f'hello_from_mmap() returned {result}')

lib = ctypes.CDLL('./libhello_mmap.so')

# The symbol's runtime address, as a plain integer
address = ctypes.cast(lib.hello_from_mmap, ctypes.c_void_p).value
print(f'hello_from_mmap at 0x{address:x} via ctypes.CDLL')

# Build a callable from that raw address and call it by pointer: int hello_from_mmap(void)
hello_from_mmap = ctypes.CFUNCTYPE(ctypes.c_int)(address)
result = hello_from_mmap()
print(f'hello_from_mmap() returned {result}')